# Fashion-MNIST: เปรียบเทียบ DNN, CNN และทดลองปรับปรุงโมเดล
รันเซลล์จากบนลงล่าง หรือ Restart Kernel แล้ว Run All เพื่อให้ตัวแปรตรงกันทั้งงาน

ลำดับงาน: ตรวจข้อมูล → แบ่ง Train/Validation → ฝึก DNN โมเดลเดิม และ CNN → เลือก โมเดลเดิมที่เลือก ด้วย Validation → เพิ่ม Dropout และ Early Stopping → เปรียบเทียบ → เลือกโมเดลสุดท้าย → ประเมิน Test → วิเคราะห์ภาพที่ผิด

**บทบาทของข้อมูล:** Training ใช้ปรับน้ำหนัก, Validation ใช้เลือกและปรับโมเดล, Test ใช้ประเมินท้ายงาน ห้ามนำคะแนน Test กลับไปจูน

ไฟล์ฉบับเรียบเรียงนี้ล้างผลรันเก่าเพื่อไม่ให้ปะปนกับโค้ดใหม่ ต้องรันเพื่อสร้างผลจริง ไม่มีการใส่คะแนนสมมติ

## ชื่อโมเดลที่ใช้ตลอดไฟล์
| ชื่อที่แสดงในผลและกราฟ | ความหมาย | ตัวแปรโมเดล |
|---|---|---|
| DNN Original | DNN โมเดลเดิม ก่อนเพิ่มเทคนิค | `dnn_original` |
| CNN Original | CNN โมเดลเดิม ก่อนเพิ่มเทคนิค | `cnn_original` |
| CNN New (Dropout + Early Stopping) หรือ DNN New (Dropout + Early Stopping) | โมเดลใหม่ สร้างตามชนิดของโมเดลเดิมที่เลือกและเพิ่มสองเทคนิค | `new_model` |

`selected_original_model` อ้างถึงโมเดลเดิมที่เลือก ไม่ใช่โมเดลเพิ่มอีกตัว ส่วน `final_model` อ้างถึงตัวที่เลือกใช้หลังเปรียบเทียบ ซึ่งอาจเป็นโมเดลเดิมหรือโมเดลใหม่ก็ได้ ชื่อกราฟใช้ Original = เดิม และ New = ใหม่ เพื่ออ่านได้แม้เครื่องไม่มีฟอนต์ไทย


## 1. เตรียมไลบรารีและค่าการทดลอง
ถ้ายังไม่มีไลบรารี ให้เอาเครื่องหมาย `#` หน้า `%pip` ออกแล้วรัน จากนั้น Restart Kernel หากจำเป็น ค่าของ Seed ช่วยให้ทำซ้ำได้ แต่ผลอาจยังต่างตามอุปกรณ์และรุ่นไลบรารี


In [ ]:
# %pip install tensorflow numpy matplotlib scikit-learn pandas


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

import pandas as pd
from IPython.display import display

SEED = 42
EPOCHS = 10
BATCH_SIZE = 64
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print(f"Seed={SEED}, Epoch limit={EPOCHS}, Batch size={BATCH_SIZE}")


## 2. โหลดและตรวจข้อมูลดิบ
หนึ่งภาพมีขนาด 28 × 28 พิกเซล และ Label เป็นเลข 0–9 ตรวจจำนวนภาพให้ตรงกับจำนวนคำตอบก่อนใช้งาน ชุด Test แยกไว้ตั้งแต่ต้น


In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = (
    tf.keras.datasets.fashion_mnist.load_data()
)

print("Training images:", x_train_full.shape)
print("Training labels:", y_train_full.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)
print("Image dtype:", x_train_full.dtype)
print("Raw pixel range:", x_train_full.min(), "to", x_train_full.max())
print("First 10 labels:", y_train_full[:10])


## 3. จับคู่เลขคลาสกับชื่อเสื้อผ้าและดูภาพตัวอย่าง
ชื่อใน `class_names` เรียงตามเลข Label ภาพต่อไปนี้เป็น 10 ภาพแรก ไม่จำเป็นต้องครบทั้ง 10 คลาส


In [ ]:
class_names = [
    "T-shirt/top",  # 0
    "Trouser",      # 1
    "Pullover",     # 2
    "Dress",        # 3
    "Coat",         # 4
    "Sandal",       # 5
    "Shirt",        # 6
    "Sneaker",      # 7
    "Bag",          # 8
    "Ankle boot"    # 9
]
display(pd.DataFrame({"Label": range(10), "Class": class_names}))


In [ ]:
plt.figure(figsize=(12, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_full[i], cmap="gray")
    plt.title(class_names[y_train_full[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()


## 4. ปรับพิกเซลเป็นช่วง 0–1
แปลงเป็น float32 แล้วหารด้วย 255 เพื่อให้ขนาดค่าข้อมูลเหมาะกับการฝึก ใช้ขั้นตอนเดียวกันกับ Train และ Test และอย่ารันเซลล์นี้ซ้ำโดยไม่โหลดข้อมูลดิบใหม่


In [ ]:
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Minimum:", x_train_full.min())
print("Maximum:", x_train_full.max())
print("Normalized dtype:", x_train_full.dtype)
print("Test range:", x_test.min(), "to", x_test.max())


## 5. แบ่ง Validation จากชุด Training เดิม
แบ่ง 20% เป็น Validation และใช้ stratify รักษาสัดส่วนคลาส จึงได้ Train 48,000, Validation 12,000 และ Test 10,000 ภาพ ตารางช่วยตรวจจำนวนแต่ละคลาส


In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train_full
)

print("Train:", x_train.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)
assert len(x_train) == len(y_train)
assert len(x_val) == len(y_val)
assert len(x_test) == len(y_test)
display(pd.DataFrame({
    "Class": class_names,
    "Train": np.bincount(y_train, minlength=10),
    "Validation": np.bincount(y_val, minlength=10),
    "Test": np.bincount(y_test, minlength=10)
}))


## 6. สร้าง DNN โมเดลเดิม
DNN รับภาพ (28, 28) แล้ว Flatten เป็นเวกเตอร์ 784 ค่า Dense เรียนรู้เพื่อทำนาย 10 คลาส ดู `summary()` เพื่อเชื่อมขนาดข้อมูลกับจำนวนพารามิเตอร์


In [ ]:
dnn_original = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
], name="dnn_original")

dnn_original.summary()


## 7. กำหนดวิธีเรียนรู้และฝึก DNN โมเดลเดิม
`compile()` ตั้ง Optimizer, Loss และตัวชี้วัด จากนั้น `fit()` ปรับน้ำหนักด้วย Training และตรวจ Validation หลังแต่ละ Epoch ใช้ sparse categorical crossentropy เพราะคำตอบเป็นเลขคลาส ไม่ใช่ one-hot


In [ ]:
dnn_original.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history_dnn_original = dnn_original.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)


## 8. แสดงผล DNN โมเดลเดิม ทุก Epoch
History เก็บผลระหว่างฝึก ไม่ใช่ตัวโมเดล แสดงเป็นตารางแทนการเขียนหลาย expression ซึ่ง Notebook จะแสดงอัตโนมัติเพียงบรรทัดสุดท้าย Accuracy ในตารางเป็นสัดส่วน เช่น 0.90 = 90%


In [ ]:
def history_table(history):
    table = pd.DataFrame(history.history)
    table.index = np.arange(1, len(table) + 1)
    table.index.name = "Epoch"
    return table

display(history_table(history_dnn_original).round(4))


## 9. เตรียมรูปทรงข้อมูลสำหรับ CNN
เพิ่มแกน Channel ขนาด 1 เพราะเป็นภาพสีเทา จาก (จำนวนภาพ, 28, 28) เป็น (จำนวนภาพ, 28, 28, 1) ภาพและคำตอบยังเป็นชุดเดียวกับ DNN


In [ ]:
x_train_cnn = x_train[..., np.newaxis]
x_val_cnn = x_val[..., np.newaxis]
x_test_cnn = x_test[..., np.newaxis]

for name, data in [("Train CNN", x_train_cnn), ("Validation CNN", x_val_cnn), ("Test CNN", x_test_cnn)]:
    print(f"{name}: {data.shape}")


## 10. สร้างและฝึก CNN โมเดลเดิม
Conv2D เรียนรู้ลักษณะเชิงพื้นที่ MaxPooling ลดขนาด Feature Map แล้ว Dense ใช้ลักษณะที่ได้จำแนกประเภท ใช้จำนวน Epoch และ Batch size เท่ากับ DNN


In [ ]:
cnn_original = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(
        32, (3, 3),
        activation="relu",
        padding="valid"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(
        64, (3, 3),
        activation="relu",
        padding="valid"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
], name="cnn_original")

cnn_original.summary()


In [ ]:
cnn_original.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_cnn_original = cnn_original.fit(
    x_train_cnn,
    y_train,
    validation_data=(x_val_cnn, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)
display(history_table(history_cnn_original).round(4))


## 11. อ่านกราฟ Accuracy และ Loss ของทั้งสองโมเดล
Accuracy สูงขึ้น = ตอบถูกมากขึ้น; Loss ต่ำลง = ความผิดพลาดตามฟังก์ชันที่ใช้ฝึกลดลง

ถ้า Training Loss ลดแต่ Validation Loss เพิ่มต่อเนื่อง เป็นสัญญาณ Overfitting ช่องว่าง Accuracy เพียงอย่างเดียวยังไม่พอสรุป ถ้าทั้งสองชุดยังดีขึ้นอาจยังฝึกต่อได้ ถ้าทั้งสองแย่และนิ่งให้ตรวจข้อมูล การฝึก และความสามารถของโมเดล


In [ ]:
def plot_histories(items):
    fig, axes = plt.subplots(2, len(items), figsize=(6 * len(items), 8), squeeze=False)
    for col, (name, history) in enumerate(items):
        epochs = range(1, len(history.history["loss"]) + 1)
        for row, metric in enumerate(["accuracy", "loss"]):
            ax = axes[row, col]
            ax.plot(epochs, history.history[metric], label="Training")
            ax.plot(epochs, history.history["val_" + metric], label="Validation")
            ax.set(title=f"{name}: {metric}", xlabel="Epoch", ylabel=metric)
            ax.legend()
            ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_histories([("DNN Original", history_dnn_original), ("CNN Original", history_cnn_original)])


## 12. เลือก โมเดลเดิมที่เลือก ด้วย Validation
ประเมินน้ำหนักปัจจุบันของ DNN และ CNN แล้วแสดงคะแนนทั้งคู่ก่อนเลือก ตัวเดิมยังไม่ได้คืนค่าน้ำหนักจาก Epoch ก่อนหน้า จึงไม่ใช้ค่าสูงสุดใน History แทนน้ำหนักปัจจุบัน

โมเดลเดิมที่เลือก หมายถึงโมเดลเดิมที่ใช้เทียบ ไม่ใช่โมเดลชนิดใหม่ `history_selected_original` เก็บประวัติ ส่วน `x_*_selected` เลือกข้อมูลให้ตรงรูปทรงของโมเดล กรณีคะแนนเท่ากันกำหนดให้เลือก CNN


In [ ]:
dnn_val_loss, dnn_val_acc = dnn_original.evaluate(x_val, y_val, verbose=0)
cnn_val_loss, cnn_val_acc = cnn_original.evaluate(x_val_cnn, y_val, verbose=0)
display(pd.DataFrame({
    "Model": ["DNN Original", "CNN Original"],
    "Validation Loss": [dnn_val_loss, cnn_val_loss],
    "Validation Accuracy (%)": [100*dnn_val_acc, 100*cnn_val_acc]
}).round(4))
use_cnn = cnn_val_acc >= dnn_val_acc
selected_original_name = "CNN Original" if use_cnn else "DNN Original"
new_model_name = "CNN New (Dropout + Early Stopping)" if use_cnn else "DNN New (Dropout + Early Stopping)"
selected_original_model = cnn_original if use_cnn else dnn_original
history_selected_original = history_cnn_original if use_cnn else history_dnn_original
x_train_selected = x_train_cnn if use_cnn else x_train
x_val_selected = x_val_cnn if use_cnn else x_val
x_test_selected = x_test_cnn if use_cnn else x_test
print("โมเดลเดิมที่เลือกมาเปรียบเทียบ:", selected_original_name)
print("Selected input shape:", x_train_selected.shape)
print(f"CNN minus DNN validation accuracy: {(cnn_val_acc-dnn_val_acc)*100:+.3f} percentage points")


## 13. ตั้งสมมติฐานและสร้าง โมเดลใหม่ (เพิ่ม Dropout + Early Stopping) แยกจาก โมเดลเดิมที่เลือก
เราจะทดลองสองเทคนิค โดยคงชั้นเดิมและข้อมูลเดิมไว้:

| เทคนิค | เปลี่ยนอะไร | เหตุผลที่ทดลอง |
|---|---|---|
| Dropout 0.3 | สุ่มปิดผลลัพธ์บางหน่วยเฉพาะตอนฝึก | ลดการพึ่งพาหน่วยบางตัวมากเกินไป เมื่อผล Training ดีขึ้นแต่ Validation ตามไม่ทัน |
| Early Stopping | หยุดเมื่อ Validation Accuracy ไม่ดีขึ้นตามเงื่อนไข | ลดการฝึกต่อที่ไม่เพิ่มคะแนน Validation และคืนค่าน้ำหนักที่ดีที่สุด |

ตรวจกราฟข้อ 11 ก่อนเขียนเหตุผลจริง ถ้าไม่มีสัญญาณ Overfitting อย่าเขียนว่าเกิดแน่นอน เทคนิคเหล่านี้เป็นการทดลอง ไม่รับประกันว่าดีขึ้น และหากใช้พร้อมกันจะยังแยกไม่ได้ว่าแต่ละวิธีช่วยเท่าไร

`new_model` เป็นโมเดลใหม่ เริ่มเรียนรู้น้ำหนักใหม่ ไม่ใช่การนำ โมเดลเดิมที่เลือก มาฝึกต่อ `Dropout(0.3)` ไม่ได้ลบ Neurons ถาวร และปิดการสุ่มเมื่อประเมิน/ทำนาย


In [ ]:
if use_cnn:
    new_model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ], name="cnn_new")
else:
    new_model = models.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ], name="dnn_new")

new_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
new_model.summary()
print("จำนวนพารามิเตอร์โมเดลเดิม:", selected_original_model.count_params())
print("จำนวนพารามิเตอร์โมเดลใหม่:", new_model.count_params())
print("Dropout does not add trainable parameters.")


## 14. ฝึก โมเดลใหม่ (เพิ่ม Dropout + Early Stopping) ด้วยข้อมูลเดิม
ติดตาม val_accuracy แบบค่าสูงดีกว่า ถ้าไม่ดีขึ้น 3 Epoch ติดต่อกันให้หยุด และคืนค่าน้ำหนักที่ดีที่สุด เพดานยังเป็น 10 Epoch เหมือนเดิม อาจฝึกครบโดยไม่หยุดก่อนก็ได้ ตาราง History ยังคงมี Epoch หลังจุดที่ดีที่สุดแม้คืนน้ำหนักแล้ว


In [ ]:
print("กำลังฝึกโมเดลใหม่:", new_model_name)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=3,
    restore_best_weights=True
)

history_new_model = new_model.fit(
    x_train_selected,
    y_train,
    validation_data=(x_val_selected, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop]
)

print(
    "Actual epochs:",
    len(history_new_model.history["loss"])
)
best_epoch = int(np.argmax(history_new_model.history["val_accuracy"])) + 1
print("Best validation-accuracy epoch:", best_epoch)
print(f"Best recorded validation accuracy: {max(history_new_model.history['val_accuracy']):.2%}")
print("Stopped before epoch limit:", len(history_new_model.history["loss"]) < EPOCHS)
display(history_table(history_new_model).round(4))


## 15. เปรียบเทียบกราฟ โมเดลเดิมที่เลือก กับ โมเดลใหม่ (เพิ่ม Dropout + Early Stopping)
ดู Validation Loss และ Accuracy ร่วมกัน อย่าตัดสินจากช่องว่างเล็กลงอย่างเดียว เพราะทั้งสองชุดอาจแย่ลงได้ Dropout ทำงานเฉพาะตอนฝึก จึงอาจเห็น Training Accuracy ต่ำกว่า Validation ได้


In [ ]:
plot_histories([(selected_original_name, history_selected_original), (new_model_name, history_new_model)])


## 16. ประเมินน้ำหนักปัจจุบัน และเลือกโมเดลสุดท้ายก่อน Test
`evaluate()` วัดผลโดยไม่ปรับน้ำหนัก ประเมินใหม่เพื่อให้คะแนนตรงกับน้ำหนักที่คืนมาจาก Early Stopping ผล Training ในตารางเป็นโหมดประเมิน (ไม่เปิด Dropout) จึงอาจต่างจาก History

เลือกคะแนน Validation สูงสุดจากทั้งสามตัว ถ้าเท่ากัน เลือก โมเดลเดิมที่เลือก ก่อน แล้วจึงตัวอื่นตามลำดับที่กำหนด


In [ ]:
candidates = {
    "DNN Original": (dnn_original, x_train, x_val, x_test),
    "CNN Original": (cnn_original, x_train_cnn, x_val_cnn, x_test_cnn),
    new_model_name: (
        new_model,
        x_train_selected,
        x_val_selected,
        x_test_selected
    )
}

results = {}

for name, (model, xtr, xv, xt) in candidates.items():
    train_loss, train_acc = model.evaluate(
        xtr, y_train, verbose=0
    )
    val_loss, val_acc = model.evaluate(
        xv, y_val, verbose=0
    )

    results[name] = {
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    }

selection_order = [selected_original_name, "DNN Original" if use_cnn else "CNN Original", new_model_name]
best_name = max(
    selection_order,
    key=lambda name: results[name]["val_acc"]
)

print("Final model selected by validation:", best_name)
validation_table = pd.DataFrame(results).T
validation_table.index.name = "Model"
display(validation_table.round(4))
original_result = results[selected_original_name]
new_result = results[new_model_name]
print(f"New model minus original model validation accuracy: {(new_result['val_acc']-original_result['val_acc'])*100:+.3f} percentage points")
print(f"New model minus original model validation loss: {new_result['val_loss']-original_result['val_loss']:+.4f}")
print("Positive accuracy difference is better; negative loss difference is better.")
final_model = candidates[best_name][0]


## 17. ประเมิน Test ในรอบสุดท้าย
เมื่อเลือกโมเดลแล้ว จึงประเมินทุกโมเดลที่ต้องรายงาน ห้ามใช้คะแนนนี้กลับไปปรับโมเดล โมเดลเดิมที่เลือก คือ DNN หรือ CNN ตัวเดิมจึงไม่ต้องประเมินซ้ำในชื่อใหม่ ตารางใช้ Accuracy เป็นสัดส่วน และแสดงผลโมเดลสุดท้ายเป็นเปอร์เซ็นต์เพิ่มเติม


In [ ]:
for name, (model, xtr, xv, xt) in candidates.items():
    test_loss, test_acc = model.evaluate(
        xt, y_test, verbose=0
    )

    results[name]["test_loss"] = test_loss
    results[name]["test_acc"] = test_acc

    print(
        f"{name}: "
        f"Train Acc={results[name]['train_acc']:.4f}, "
        f"Val Acc={results[name]['val_acc']:.4f}, "
        f"Test Acc={test_acc:.4f}, "
        f"Test Loss={test_loss:.4f}"
    )
results_table = pd.DataFrame(results).T
results_table.index.name = "Model"
display(results_table[["train_acc", "val_acc", "test_acc", "train_loss", "val_loss", "test_loss"]].round(4))
print(f"Final selected model: {best_name}; test accuracy = {results[best_name]['test_acc']:.2%}")
print("โมเดลเดิมที่เลือก เทียบกับ โมเดลใหม่:")
display(results_table.loc[[selected_original_name, new_model_name]].round(4))


## 18. ดูว่า DNN และ CNN สับสนคลาสไหน
`predict()` ให้คะแนน 10 คลาสต่อภาพ `argmax(axis=1)` เลือกเลขคลาสที่คะแนนสูงที่สุดในแต่ละแถว

Confusion Matrix: แถว = คำตอบจริง คอลัมน์ = คำทำนาย แนวทแยง = ถูก นอกแนวทแยง = ผิด เช่น แถว Shirt คอลัมน์ T-shirt/top หมายถึง Shirt ถูกทำนายเป็น T-shirt/top


In [ ]:
predictions = {}
probabilities_by_model = {}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, name in zip(axes, ["DNN Original", "CNN Original"]):
    model, _, _, xt = candidates[name]

    probabilities = model.predict(xt, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)

    predictions[name] = y_pred
    probabilities_by_model[name] = probabilities
    print(f"{name}: probabilities shape={probabilities.shape}, predictions shape={y_pred.shape}")
    print(f"Correct={np.sum(y_pred == y_test)}, Incorrect={np.sum(y_pred != y_test)}")

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        labels=np.arange(10),
        display_labels=class_names,
        cmap="Blues",
        xticks_rotation=90,
        colorbar=False,
        ax=ax
    )

    ax.set_title(name)

plt.tight_layout()
plt.show()


In [ ]:
# ตัวอย่างหนึ่งภาพ เพื่อเชื่อมคะแนน -> เลขคลาส -> ชื่อคลาส
sample_index = 0
for name in ["DNN Original", "CNN Original"]:
    print(f"{name}, image {sample_index}, true class: {class_names[y_test[sample_index]]}")
    display(pd.DataFrame({"Class": class_names, "Score": probabilities_by_model[name][sample_index]}).round(4))
    print("Predicted class:", class_names[predictions[name][sample_index]])


## 19. แสดงภาพถูก 5 ภาพและผิด 5 ภาพ
เลือก DNN หรือ CNN ที่ Test Accuracy สูงกว่าเพื่อแสดงตัวอย่างตามขั้นตอนงานเดิม การเลือกส่วนนี้ไม่เปลี่ยนโมเดลสุดท้ายที่เลือกด้วย Validation

`np.where()` หาตำแหน่งที่ทำนายตรง/ไม่ตรงกับคำตอบจริง แล้ว `[:5]` เลือกห้าภาพแรก ตัวอย่างเหล่านี้ไม่ได้แทนความผิดพลาดทั้งหมด


In [ ]:
example_name = max(
    ["DNN Original", "CNN Original"],
    key=lambda name: results[name]["test_acc"]
)

y_pred = predictions[example_name]

correct_indices = np.where(y_pred == y_test)[0][:5]
wrong_indices = np.where(y_pred != y_test)[0][:5]

print("Model used for examples:", example_name)
print("Correct image indices:", correct_indices)
print("Incorrect image indices:", wrong_indices)
example_indices = np.concatenate([correct_indices, wrong_indices])
display(pd.DataFrame({
    "Image index": example_indices,
    "True class": [class_names[y_test[i]] for i in example_indices],
    "Predicted class": [class_names[y_pred[i]] for i in example_indices],
    "Correct": y_pred[example_indices] == y_test[example_indices]
}))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for row, indices in enumerate(
    [correct_indices, wrong_indices]
):
    for col in range(5):
        ax = axes[row, col]
        ax.axis("off")

        if col >= len(indices):
            continue

        idx = indices[col]

        ax.imshow(x_test[idx], cmap="gray")
        ax.set_title(
            f"True: {class_names[y_test[idx]]}\n"
            f"Pred: {class_names[y_pred[idx]]}",
            color="green" if row == 0 else "red",
            fontsize=9
        )

plt.suptitle(example_name)
plt.tight_layout()
plt.show()


## 20. เขียนอภิปรายจากผลจริงหลังรัน

| คำถาม | หลักฐานที่จะอ้าง |
|---|---|
| ทำไมเลือก โมเดลเดิมที่เลือก ตัวนี้? | ตาราง Validation ข้อ 12 |
| ปรับอะไรและเพราะอะไร? | กราฟข้อ 11 และสมมติฐานข้อ 13 |
| โมเดลใหม่ (เพิ่ม Dropout + Early Stopping) ดีขึ้นหรือไม่? | ผลต่าง Validation และตารางข้อ 16 |
| ลด Overfitting หรือไม่? | แนวโน้ม Training/Validation Loss ข้อ 15 |
| โมเดลสุดท้ายคืออะไรและ Test ได้เท่าไร? | การเลือกข้อ 16 และผลข้อ 17 |
| โมเดลสับสนอะไร? | Confusion Matrix ข้อ 18 |
| ภาพที่ผิดมีลักษณะอย่างไร? | ตัวอย่างภาพข้อ 19 |

เติมข้อสรุปของตนเองหลังรัน: **[โมเดลเดิมที่เลือก และเหตุผล] → [เทคนิคที่ทดลอง] → [ผลต่างที่ได้จริง] → [โมเดลสุดท้ายและ Test] → [ข้อผิดพลาดที่ยังพบ]**

อย่าสรุปว่าเทคนิคช่วยแน่นอนจากชื่อ โมเดลใหม่ (เพิ่ม Dropout + Early Stopping) หรือจาก Training Accuracy อย่างเดียว ผลการรันครั้งเดียวอาจมีความแปรปรวนจากการเริ่มน้ำหนักและการสุ่ม และภาพ Fashion-MNIST ขนาดเล็กสีเทายังไม่ใช่หลักฐานว่าใช้กับภาพสินค้าจริงได้ทันที
